# Direct Bayesian 10 -> 150 PCOS ECG Synthetic Data Notebook

This notebook trains directly on `PCOS_ECG_Centric_16_Features_Corrected.csv`, generates five 150-row Bayesian candidates, validates them, ranks them, and certifies the best dataset. It does not perform recursive synthesis.

In [ ]:

# ================================================================
# Direct Bayesian Network Synthesis: PCOS ECG-Centric Dataset
# ================================================================
# This Google Colab notebook trains only on the original corrected
# 10-subject dataset and generates 150 synthetic samples directly.
#
# IMPORTANT:
# - No recursive synthesis is used.
# - Subject_ID is treated as an identifier and excluded from quality metrics.
# - Five independent candidate datasets are generated and ranked.
# - Certification is run only on the best-ranked candidate.
# ================================================================

# %%capture
%pip install -q -U sdv sdmetrics pandas numpy scipy scikit-learn matplotlib seaborn networkx
%pip install -q -U umap-learn

import os
import sys
import json
import math
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from scipy.stats import ks_2samp, wasserstein_distance, zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

import sdv
try:
    from sdv.metadata import Metadata
except Exception:
    from sdv.metadata import SingleTableMetadata
    Metadata = None

from sdmetrics.reports.single_table import QualityReport

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SEEDS = [42, 123, 456, 789, 999]
N_SYNTHETIC = 150
TABLE_NAME = "pcos_ecg"

INPUT_FILE = "PCOS_ECG_Centric_16_Features_Corrected.csv"
IDENTIFIER = "Subject_ID"
DECLARED_CATEGORICAL_COLUMNS = [
    "Diagnosis",
    "Target_Class",
    "Cycle Regularity",
    "Family History (PCOS/Diabetes)",
]
PRIMARY_CLASS_COLUMN = "Target_Class"

OUTPUT_ROOT = Path("outputs")
DATASET_DIR = OUTPUT_ROOT / "datasets"
REPORT_DIR = OUTPUT_ROOT / "reports"
PLOT_DIR = OUTPUT_ROOT / "plots"
VALIDATION_DIR = OUTPUT_ROOT / "validation"

for directory in [OUTPUT_ROOT, DATASET_DIR, REPORT_DIR, PLOT_DIR, VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
})

def reset_seeds(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

reset_seeds()
print(f"Python: {sys.version.split()[0]}")
print(f"SDV: {getattr(sdv, '__version__', 'installed')}")
print(f"UMAP available: {UMAP_AVAILABLE}")


# ================================================================
# Data Loading and Audit
# ================================================================

def locate_input_file(filename=INPUT_FILE):
    candidates = [
        Path(filename),
        Path("/content") / filename,
        Path("unsorted outputs") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} was not found. In Colab, upload it to /content before running the notebook."
    )

def clean_column_names(frame):
    frame = frame.copy()
    frame.columns = [str(c).strip().replace("\n", " ").replace("\r", " ") for c in frame.columns]
    return frame

input_path = locate_input_file()
original_raw = clean_column_names(pd.read_csv(input_path))

if IDENTIFIER not in original_raw.columns:
    original_raw.insert(0, IDENTIFIER, [f"S{i+1:02d}" for i in range(len(original_raw))])

categorical_columns = [c for c in DECLARED_CATEGORICAL_COLUMNS if c in original_raw.columns]
object_columns = [
    c for c in original_raw.columns
    if c != IDENTIFIER and c not in categorical_columns and original_raw[c].dtype == "object"
]
if object_columns:
    print("Additional non-numeric columns treated as categorical for executable synthesis:", object_columns)
categorical_columns = categorical_columns + object_columns

numeric_columns = [
    c for c in original_raw.columns
    if c not in categorical_columns + [IDENTIFIER]
]

original = original_raw.copy()
for column in numeric_columns:
    original[column] = pd.to_numeric(original[column], errors="coerce")
    original[column] = original[column].fillna(original[column].median())
for column in categorical_columns:
    original[column] = original[column].astype("string").fillna("Missing").astype(str)
original[IDENTIFIER] = original[IDENTIFIER].astype(str)

print("Original Samples:", len(original))
print("Shape:", original.shape)
print("\nData Types:")
display(original.dtypes.to_frame("dtype"))
print("\nMissing Value Report:")
display(original.isna().sum().to_frame("missing_values"))
print("\nSummary Statistics:")
display(original.describe(include="all").T)
print("\nClass Distribution:")
if PRIMARY_CLASS_COLUMN in original.columns:
    display(original[PRIMARY_CLASS_COLUMN].value_counts(dropna=False).to_frame("count"))
else:
    print(f"{PRIMARY_CLASS_COLUMN} not found.")

original.to_csv(REPORT_DIR / "original_dataset_audit.csv", index=False)
original.describe(include="all").T.to_csv(REPORT_DIR / "original_summary_statistics.csv")
original.isna().sum().to_frame("missing_values").to_csv(REPORT_DIR / "original_missing_value_report.csv")


# ================================================================
# Metadata for SDV Quality Reports
# Subject_ID is excluded from SDV quality evaluation.
# ================================================================

metric_columns = [c for c in original.columns if c != IDENTIFIER]
metric_original = original[metric_columns].copy()

def build_quality_metadata(frame, categorical):
    if Metadata is not None:
        metadata = Metadata.detect_from_dataframe(data=frame, table_name=TABLE_NAME)
        for column in frame.columns:
            if column in categorical:
                metadata.update_column(column_name=column, sdtype="categorical")
            else:
                metadata.update_column(column_name=column, sdtype="numerical")
        return metadata.to_dict()["tables"][TABLE_NAME]
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(frame)
    for column in frame.columns:
        if column in categorical:
            metadata.update_column(column_name=column, sdtype="categorical")
        else:
            metadata.update_column(column_name=column, sdtype="numerical")
    return metadata.to_dict()

quality_metadata = build_quality_metadata(metric_original, categorical_columns)


# ================================================================
# Publication-Quality Baseline Plots
# ================================================================

def save_correlation_heatmap(corr, title, path):
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="vlag", center=0, square=True, linewidths=0.25, cbar_kws={"shrink": 0.75})
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

original_corr = original[numeric_columns].corr(method="pearson")
original_corr.to_csv(REPORT_DIR / "original_correlation_matrix.csv")
save_correlation_heatmap(original_corr, "Original Pearson Correlation Matrix", PLOT_DIR / "original_correlation_matrix.png")

def plot_feature_distributions(real, synthetic=None, label="original", path_prefix=PLOT_DIR / "feature_distributions"):
    path_prefix.parent.mkdir(parents=True, exist_ok=True)
    n = len(numeric_columns)
    cols = 4
    rows = int(math.ceil(n / cols))
    plt.figure(figsize=(4 * cols, 3 * rows))
    for i, column in enumerate(numeric_columns, start=1):
        plt.subplot(rows, cols, i)
        sns.kdeplot(real[column], label="Original", fill=True, alpha=0.25)
        if synthetic is not None:
            sns.kdeplot(synthetic[column], label="Synthetic", fill=True, alpha=0.25)
        plt.title(column, fontsize=9)
        plt.xlabel("")
    handles, labels = plt.gca().get_legend_handles_labels()
    if handles:
        plt.figlegend(handles, labels, loc="upper right")
    plt.tight_layout()
    out = Path(f"{path_prefix}_{label}.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

plot_feature_distributions(original, label="original")

def plot_pca_comparison(real, synthetic=None, title="PCA Visualization", path=PLOT_DIR / "pca_original.png"):
    if synthetic is None:
        combined = real[numeric_columns].copy()
        source = np.array(["Original"] * len(real))
    else:
        combined = pd.concat([real[numeric_columns], synthetic[numeric_columns]], ignore_index=True)
        source = np.array(["Original"] * len(real) + ["Synthetic"] * len(synthetic))
    scaled = StandardScaler().fit_transform(combined)
    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(scaled)
    pca_df = pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1], "Source": source})
    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Source", s=65, alpha=0.85, edgecolor="black", linewidth=0.3)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

plot_pca_comparison(original)


# ================================================================
# Direct BayesianNetworkSynthesizer
# ================================================================
# This implementation uses a Bayesian-network-style conditional
# generative model:
# - Target_Class is the root and is sampled with exact original balance.
# - Categorical variables are sampled from empirical conditional tables.
# - Numerical variables are sampled from class-conditional Gaussian
#   networks with shrinkage covariance for stability on n=10.
# - Derived ECG ratio/product columns are repaired after sampling.
# ================================================================

class BayesianNetworkSynthesizer:
    def __init__(self, categorical_columns, numeric_columns, class_column="Target_Class", id_column="Subject_ID", random_state=42):
        self.categorical_columns = list(categorical_columns)
        self.numeric_columns = list(numeric_columns)
        self.class_column = class_column
        self.id_column = id_column
        self.random_state = random_state
        self.fitted = False

    def fit(self, data):
        self.data = data.copy()
        self.rng = np.random.default_rng(self.random_state)
        self.class_counts = self.data[self.class_column].value_counts().to_dict()
        self.class_values = list(self.class_counts.keys())
        self.global_means = self.data[self.numeric_columns].mean()
        self.global_cov = self._stable_cov(self.data[self.numeric_columns])
        self.bounds = self.data[self.numeric_columns].agg(["min", "max"])
        self.decimals = {
            c: max([self._count_decimals(v) for v in self.data[c].dropna().head(10)] + [0])
            for c in self.numeric_columns
        }
        self.class_params = {}
        for cls in self.class_values:
            subset = self.data[self.data[self.class_column] == cls]
            if len(subset) >= 2:
                self.class_params[cls] = (subset[self.numeric_columns].mean(), self._stable_cov(subset[self.numeric_columns]))
            else:
                self.class_params[cls] = (self.global_means, self.global_cov)
        self.cat_tables = {}
        for column in self.categorical_columns:
            if column == self.class_column:
                continue
            self.cat_tables[column] = {}
            global_probs = self.data[column].value_counts(normalize=True)
            for cls in self.class_values:
                subset = self.data[self.data[self.class_column] == cls]
                probs = subset[column].value_counts(normalize=True)
                if probs.empty:
                    probs = global_probs
                self.cat_tables[column][cls] = probs
        self.fitted = True
        return self

    def _count_decimals(self, value):
        try:
            text = f"{float(value):.10f}".rstrip("0").rstrip(".")
            return len(text.split(".")[1]) if "." in text else 0
        except Exception:
            return 3

    def _stable_cov(self, frame):
        values = frame.to_numpy(dtype=float)
        if values.shape[0] < 2:
            cov = np.diag(np.maximum(np.abs(values[0]) * 0.02, 1e-4))
        else:
            cov = np.cov(values, rowvar=False)
        cov = np.atleast_2d(cov)
        diag = np.diag(np.diag(cov))
        cov = 0.70 * cov + 0.30 * diag
        cov = cov + np.eye(cov.shape[0]) * 1e-6
        return cov

    def _balanced_classes(self, num_rows):
        total = sum(self.class_counts.values())
        raw = {cls: num_rows * count / total for cls, count in self.class_counts.items()}
        counts = {cls: int(math.floor(v)) for cls, v in raw.items()}
        remainder = num_rows - sum(counts.values())
        order = sorted(raw, key=lambda cls: raw[cls] - counts[cls], reverse=True)
        for cls in order[:remainder]:
            counts[cls] += 1
        classes = []
        for cls, count in counts.items():
            classes.extend([cls] * count)
        self.rng.shuffle(classes)
        return classes

    def _sample_category(self, probs):
        values = probs.index.to_numpy()
        p = probs.to_numpy(dtype=float)
        p = p / p.sum()
        return self.rng.choice(values, p=p)

    def _repair_derived_features(self, row):
        def has(*cols):
            return all(c in row.index for c in cols)
        if has("HRV_SDNN_Estimate (ms)", "Mean_RR_Interval (s)", "SDNN_MeanRR_Ratio"):
            if row["Mean_RR_Interval (s)"] != 0:
                row["SDNN_MeanRR_Ratio"] = row["HRV_SDNN_Estimate (ms)"] / row["Mean_RR_Interval (s)"]
        if has("QTc_Bazett (s)", "Mean_RR_Interval (s)", "QTc_MeanRR_Ratio"):
            if row["Mean_RR_Interval (s)"] != 0:
                row["QTc_MeanRR_Ratio"] = row["QTc_Bazett (s)"] / row["Mean_RR_Interval (s)"]
        if has("P_Wave_Dispersion (s)", "QTc_Bazett (s)", "PWaveDisp_x_QTc"):
            row["PWaveDisp_x_QTc"] = row["P_Wave_Dispersion (s)"] * row["QTc_Bazett (s)"]
        if has("HRV_SDNN_Estimate (ms)", "QTc_Bazett (s)", "HRVSDNN_x_QTc"):
            row["HRVSDNN_x_QTc"] = row["HRV_SDNN_Estimate (ms)"] * row["QTc_Bazett (s)"]
        return row

    def sample(self, num_rows, seed=None):
        if not self.fitted:
            raise RuntimeError("Fit the synthesizer before sampling.")
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        classes = self._balanced_classes(num_rows)
        rows = []
        for i, cls in enumerate(classes, start=1):
            means, cov = self.class_params[cls]
            numeric = self.rng.multivariate_normal(means.to_numpy(dtype=float), cov)
            row = pd.Series(numeric, index=self.numeric_columns)
            for column in self.numeric_columns:
                row[column] = np.clip(row[column], self.bounds.loc["min", column], self.bounds.loc["max", column])
            row = self._repair_derived_features(row)
            record = row.to_dict()
            for column in self.categorical_columns:
                if column == self.class_column:
                    record[column] = cls
                else:
                    record[column] = self._sample_category(self.cat_tables[column][cls])
            record[self.id_column] = f"SYN_{i:04d}"
            rows.append(record)
        synthetic = pd.DataFrame(rows)
        synthetic = synthetic[self.data.columns]
        for column in self.numeric_columns:
            synthetic[column] = synthetic[column].round(self.decimals[column])
        return synthetic


# ================================================================
# Validation Helpers
# ================================================================

def run_quality_report(real, synthetic, seed):
    report = QualityReport()
    synthetic_metric = synthetic[metric_columns].copy()
    report.generate(real[metric_columns], synthetic_metric, quality_metadata, verbose=False)
    properties = report.get_properties()
    properties.to_csv(REPORT_DIR / f"quality_properties_seed{seed}.csv", index=False)
    try:
        details = report.get_details(property_name="Column Shapes")
        details.to_csv(REPORT_DIR / f"quality_column_shapes_details_seed{seed}.csv", index=False)
    except Exception as exc:
        print(f"Column Shapes detail unavailable for seed {seed}: {exc}")
    try:
        details = report.get_details(property_name="Column Pair Trends")
        details.to_csv(REPORT_DIR / f"quality_column_pair_trends_details_seed{seed}.csv", index=False)
    except Exception as exc:
        print(f"Column Pair Trends detail unavailable for seed {seed}: {exc}")
    score_lookup = dict(zip(properties["Property"], properties["Score"]))
    return {
        "Quality Score": float(report.get_score()),
        "Column Shapes Score": float(score_lookup.get("Column Shapes", np.nan)),
        "Column Pair Trends Score": float(score_lookup.get("Column Pair Trends", np.nan)),
    }

def distribution_metrics(real, synthetic, seed):
    rows = []
    for column in numeric_columns:
        ks = ks_2samp(real[column], synthetic[column])
        wass = wasserstein_distance(real[column], synthetic[column])
        rows.append({
            "Feature": column,
            "KS Statistic": float(ks.statistic),
            "KS p-value": float(ks.pvalue),
            "Wasserstein Distance": float(wass),
        })
    result = pd.DataFrame(rows)
    result.to_csv(VALIDATION_DIR / f"distribution_preservation_seed{seed}.csv", index=False)
    return result

def correlation_metrics(real, synthetic, seed):
    real_corr = real[numeric_columns].corr(method="pearson")
    synth_corr = synthetic[numeric_columns].corr(method="pearson")
    diff = (real_corr - synth_corr).abs()
    values = diff.to_numpy()
    upper = values[np.triu_indices_from(values, k=1)]
    real_corr.to_csv(VALIDATION_DIR / f"original_correlation_matrix_seed{seed}.csv")
    synth_corr.to_csv(VALIDATION_DIR / f"synthetic_correlation_matrix_seed{seed}.csv")
    diff.to_csv(VALIDATION_DIR / f"correlation_difference_matrix_seed{seed}.csv")
    save_correlation_heatmap(synth_corr, f"Synthetic Correlation Matrix - Seed {seed}", PLOT_DIR / f"synthetic_correlation_matrix_seed{seed}.png")
    save_correlation_heatmap(diff, f"Correlation Difference Matrix - Seed {seed}", PLOT_DIR / f"correlation_difference_matrix_seed{seed}.png")
    return real_corr, synth_corr, diff, float(np.nanmean(upper)), float(np.nanmax(upper))

def find_column(options):
    for option in options:
        if option in numeric_columns:
            return option
    return None

ecg_relationship_pairs = [
    ("HRV_SDNN", "Mean RR", ["HRV_SDNN_Estimate (ms)", "HRV_SDNN"], ["Mean_RR_Interval (s)", "Mean RR"]),
    ("Mean RR", "QTc", ["Mean_RR_Interval (s)", "Mean RR"], ["QTc_Bazett (s)", "QTc"]),
    ("QTc", "PR Interval", ["QTc_Bazett (s)", "QTc"], ["PR_Interval (s)", "PR Interval"]),
    ("QTc", "T_Wave_Duration", ["QTc_Bazett (s)", "QTc"], ["T_Wave_Duration (s)", "T_Wave_Duration"]),
    ("P_Wave_Dispersion", "PR Interval", ["P_Wave_Dispersion (s)", "P_Wave_Dispersion"], ["PR_Interval (s)", "PR Interval"]),
]

def ecg_relationship_validation(real, synthetic, seed):
    rows = []
    for left_label, right_label, left_options, right_options in ecg_relationship_pairs:
        left = find_column(left_options)
        right = find_column(right_options)
        if left is None or right is None:
            continue
        original_value = real[[left, right]].corr().iloc[0, 1]
        synthetic_value = synthetic[[left, right]].corr().iloc[0, 1]
        rows.append({
            "Relationship": f"{left_label} ↔ {right_label}",
            "Feature 1": left,
            "Feature 2": right,
            "Original Correlation": float(original_value),
            "Synthetic Correlation": float(synthetic_value),
            "Absolute Difference": float(abs(original_value - synthetic_value)),
        })
    result = pd.DataFrame(rows)
    result.to_csv(VALIDATION_DIR / f"ecg_relationship_validation_seed{seed}.csv", index=False)
    return result

def outlier_report(synthetic, seed):
    iqr_rows = []
    z_rows = []
    for column in numeric_columns:
        q1 = synthetic[column].quantile(0.25)
        q3 = synthetic[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (synthetic[column] < lower) | (synthetic[column] > upper)
        if mask.any():
            iqr_rows.append({"Feature": column, "Number of Outliers": int(mask.sum()), "Method": "IQR"})
        z = np.abs(zscore(synthetic[column], nan_policy="omit"))
        z_mask = z > 3
        if np.asarray(z_mask).any():
            z_rows.append({"Feature": column, "Number of Outliers": int(np.asarray(z_mask).sum()), "Method": "Z-Score"})
    iqr_df = pd.DataFrame(iqr_rows)
    z_df = pd.DataFrame(z_rows)
    combined = pd.concat([iqr_df, z_df], ignore_index=True)
    if combined.empty:
        combined = pd.DataFrame(columns=["Feature", "Number of Outliers", "Method"])
    combined.to_csv(VALIDATION_DIR / f"outlier_report_seed{seed}.csv", index=False)
    return combined

def plot_umap_comparison(real, synthetic, seed):
    if not UMAP_AVAILABLE:
        return None
    combined = pd.concat([real[numeric_columns], synthetic[numeric_columns]], ignore_index=True)
    source = ["Original"] * len(real) + ["Synthetic"] * len(synthetic)
    scaled = StandardScaler().fit_transform(combined)
    n_neighbors = max(2, min(10, len(combined) - 1))
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=0.15, random_state=RANDOM_STATE)
    coords = reducer.fit_transform(scaled)
    plot_df = pd.DataFrame({"UMAP1": coords[:, 0], "UMAP2": coords[:, 1], "Source": source})
    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(data=plot_df, x="UMAP1", y="UMAP2", hue="Source", s=65, alpha=0.85, edgecolor="black", linewidth=0.3)
    plt.title(f"UMAP Comparison - Seed {seed}")
    plt.tight_layout()
    path = PLOT_DIR / f"umap_comparison_seed{seed}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


# ================================================================
# Generate and Validate Five Independent Candidate Datasets
# ================================================================

candidate_rows = []
candidate_artifacts = {}

for seed in SEEDS:
    print(f"\n===== Training BayesianNetworkSynthesizer and generating seed {seed} =====")
    reset_seeds(seed)
    synthesizer = BayesianNetworkSynthesizer(
        categorical_columns=categorical_columns,
        numeric_columns=numeric_columns,
        class_column=PRIMARY_CLASS_COLUMN,
        id_column=IDENTIFIER,
        random_state=seed,
    )
    synthesizer.fit(original)
    synthetic = synthesizer.sample(N_SYNTHETIC, seed=seed)
    dataset_path = DATASET_DIR / f"bayesian_150_seed{seed}.csv"
    synthetic.to_csv(dataset_path, index=False)
    print(f"Saved: {dataset_path}")
    display(synthetic.head())
    print("Synthetic class distribution:")
    display(synthetic[PRIMARY_CLASS_COLUMN].value_counts().to_frame("count"))

    quality = run_quality_report(original, synthetic, seed)
    dist = distribution_metrics(original, synthetic, seed)
    real_corr, synth_corr, corr_diff, mean_corr_error, max_corr_error = correlation_metrics(original, synthetic, seed)
    ecg_rel = ecg_relationship_validation(original, synthetic, seed)
    outliers = outlier_report(synthetic, seed)
    plot_pca_comparison(original, synthetic, f"PCA Comparison - Seed {seed}", PLOT_DIR / f"pca_comparison_seed{seed}.png")
    plot_umap_comparison(original, synthetic, seed)
    plot_feature_distributions(original, synthetic, label=f"seed{seed}")

    print("ECG relationship validation:")
    display(ecg_rel)
    print("Outlier report. Records are reported only; no records are removed.")
    display(outliers)

    row = {
        "Seed": seed,
        **quality,
        "Mean Correlation Error": mean_corr_error,
        "Maximum Correlation Error": max_corr_error,
        "Mean KS Statistic": float(dist["KS Statistic"].mean()),
        "Mean Wasserstein Distance": float(dist["Wasserstein Distance"].mean()),
    }
    candidate_rows.append(row)
    candidate_artifacts[seed] = {
        "synthetic": synthetic,
        "distribution": dist,
        "correlation_difference": corr_diff,
        "ecg_relationship": ecg_rel,
    }

seed_comparison = pd.DataFrame(candidate_rows)
ranking = seed_comparison.sort_values(
    by=["Quality Score", "Mean Correlation Error", "Mean KS Statistic"],
    ascending=[False, True, True],
).reset_index(drop=True)

ranking.to_csv(REPORT_DIR / "seed_comparison_table.csv", index=False)
display(ranking)

best_seed = int(ranking.loc[0, "Seed"])
best = candidate_artifacts[best_seed]["synthetic"]
best.to_csv(DATASET_DIR / "best_bayesian_150.csv", index=False)


# ================================================================
# Certification for Best Dataset Only
# ================================================================

best_metrics = ranking.loc[0].to_dict()
best_dist = candidate_artifacts[best_seed]["distribution"]
best_corr_diff = candidate_artifacts[best_seed]["correlation_difference"]
best_ecg = candidate_artifacts[best_seed]["ecg_relationship"]

quality_report = ranking.loc[[0], [
    "Seed",
    "Quality Score",
    "Column Shapes Score",
    "Column Pair Trends Score",
]]
quality_report.to_csv(REPORT_DIR / "quality_report.csv", index=False)

original_corr.to_csv(REPORT_DIR / "correlation_matrix.csv")
best_corr_diff.to_csv(REPORT_DIR / "correlation_difference_matrix.csv")
best_dist[["Feature", "KS Statistic", "KS p-value"]].to_csv(REPORT_DIR / "ks_results.csv", index=False)
best_dist[["Feature", "Wasserstein Distance"]].to_csv(REPORT_DIR / "wasserstein_results.csv", index=False)
best_ecg.to_csv(REPORT_DIR / "ecg_relationship_validation.csv", index=False)

ecg_relationship_preservation = (
    "PASS"
    if not best_ecg.empty and float(best_ecg["Absolute Difference"].mean()) <= 0.20
    else "REVIEW"
)

approved = (
    best_metrics["Quality Score"] >= 0.80
    and best_metrics["Mean Correlation Error"] <= 0.15
    and best_metrics["Mean KS Statistic"] <= 0.20
)
certification_status = "APPROVED" if approved else "REJECTED"

certification = pd.DataFrame([{
    "Best Seed": best_seed,
    "Quality Score": best_metrics["Quality Score"],
    "Mean Correlation Error": best_metrics["Mean Correlation Error"],
    "Mean KS Statistic": best_metrics["Mean KS Statistic"],
    "Mean Wasserstein Distance": best_metrics["Mean Wasserstein Distance"],
    "ECG Relationship Preservation": ecg_relationship_preservation,
    "Certification Status": certification_status,
    "Criterion Quality Score >= 0.80": best_metrics["Quality Score"] >= 0.80,
    "Criterion Mean Correlation Error <= 0.15": best_metrics["Mean Correlation Error"] <= 0.15,
    "Criterion Mean KS Statistic <= 0.20": best_metrics["Mean KS Statistic"] <= 0.20,
}])

certification.to_csv(REPORT_DIR / "dataset_certification_report.csv", index=False)
certification.to_csv(VALIDATION_DIR / "dataset_certification_report.csv", index=False)
display(certification)


# ================================================================
# Final Console Summary
# ================================================================

files_generated = [
    DATASET_DIR / "best_bayesian_150.csv",
    REPORT_DIR / "dataset_certification_report.csv",
    REPORT_DIR / "quality_report.csv",
    REPORT_DIR / "correlation_matrix.csv",
    REPORT_DIR / "correlation_difference_matrix.csv",
    REPORT_DIR / "ks_results.csv",
    REPORT_DIR / "wasserstein_results.csv",
    REPORT_DIR / "ecg_relationship_validation.csv",
    REPORT_DIR / "seed_comparison_table.csv",
]
files_generated += [DATASET_DIR / f"bayesian_150_seed{seed}.csv" for seed in SEEDS]

print("\n================ FINAL SUMMARY ================")
print(f"Original Samples: {len(original)}")
print(f"Generated Samples: {N_SYNTHETIC}")
print(f"Best Seed: {best_seed}")
print(f"Quality Score: {best_metrics['Quality Score']:.4f}")
print(f"Mean Correlation Error: {best_metrics['Mean Correlation Error']:.4f}")
print(f"Mean KS Statistic: {best_metrics['Mean KS Statistic']:.4f}")
print(f"Certification Status: {certification_status}")
print("\nFiles Generated:")
for path in files_generated:
    print(f"- {path}")

print("\nAll requested direct Bayesian 10 -> 150 synthesis outputs are complete.")
